In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import bidsio
import sys
import copy
import pickle
sys.path.append('../')
from helpers import *
from fns_ingp import *


device = 'cuda:5' if torch.cuda.is_available() else 'cpu'
seed, RES = 0, 64
bids_loader = bidsio.BIDSLoader(data_entities=[{'subject': '',
                                              'session': '',
                                              'suffix': 'T1w',
                                              'space': 'MNI152NLin2009aSym'}],
                              target_entities=[],
                              data_derivatives_names=['ATLAS'],
                              batch_size=1,
                              root_dir='./atlas/data/test/')


In [ ]:
idx = 131
mask = np.fft.fftshift(np.ones((RES, RES, RES))).astype(np.complex64)
tmp = bids_loader.load_sample(idx = idx, data_only=True) / 255.0
signal = resize(tmp, (1, RES, RES, RES))[0] 
plt.imshow(signal[:,:,24])
plt.colorbar()
plt.show()

In [ ]:
learning_rate, iters = 8e-4, 4000
slice_idx = 24  

# varying hash_table_size
model_params = {
    'num_levels': 4,
    'hash_table_size': 0,
    'base_resolution': 16,
    'n_features_per_level': 2.0,
}
param_vals = [13, 14]

# varying num_levels
# model_params = {
#     'num_levels': 0,
#     'hash_table_size': 16,
#     'base_resolution': 16,
#     'n_features_per_level': 2.0,
# }
# param_vals = [2, 3]

if model_params['num_levels'] == 0:
    param_to_vary = 'num_levels'
else:    
    param_to_vary = 'hash_table_size'
outputs = {}
to_save_outputs = {}
for param_val in param_vals:
    model_params[param_to_vary] = param_val
    print(f'{param_to_vary}: {model_params[param_to_vary]}')
    model_config = (int(model_params['num_levels']), int(model_params['hash_table_size']), model_params['base_resolution'], model_params['n_features_per_level'])
    output = fit_instant_ngp(signal, model_config, iters=iters, learning_rate=learning_rate, log_interval = 1000, seed=seed, device=device, mask=mask, count_params=True)
    outputs[f'{model_params[param_to_vary]}'] = output     
    to_save_outputs[f'{model_params[param_to_vary]}'] = output['best_pred'].reshape((RES, RES, RES))[:,:,slice_idx]
    error = np.linalg.norm(signal.flatten() - output['best_pred'].flatten())                       
    print(f"Error: {error:.3e}, Loss: {output['best_loss']:.3e}")

with open(f"3d_mri/ingp.pkl", "wb") as f:
    pickle.dump(to_save_outputs, f)

In [ ]:
from helpers import *

slice_idx = 24 
def slice_outputs(outputs, slice_idx):
    new_outputs = copy.deepcopy(outputs)
    small_param, large_param = list(outputs.keys())
    small_pred, large_pred   = new_outputs[small_param]['best_pred'].reshape((RES, RES, RES)), new_outputs[large_param]['best_pred'].reshape((RES, RES, RES))
    new_outputs[small_param], new_outputs[large_param] = new_outputs[small_param], new_outputs[large_param]
    new_outputs[small_param]['best_pred'] = small_pred[:,:,slice_idx]
    new_outputs[large_param]['best_pred'] = large_pred[:,:,slice_idx]
    return new_outputs

plot_error_heatmaps(signal[:,:,slice_idx], slice_outputs(outputs, slice_idx), model_name="Instant-NGP")